# 08 — Classification Fine-Tuning

**Goal:** Fine-tune `BertForSequenceClassification` on the CTI report dataset from notebook 7. By the end you'll have a saved classifier plus a per-class metrics breakdown.

## What you'll learn

1. Load the prepared dataset
2. Configure `Trainer` for sequence classification
3. Compute accuracy + per-class F1
4. Save and reload the model
5. Predict categories for new CTI text with confidence scores

## Step 1 — Load the saved data

In [1]:
import json
from datasets import load_from_disk

tokenized_ds = load_from_disk("./cls-data/tokenized")
with open("./cls-data/label_map.json") as f:
    maps = json.load(f)
label2id = maps["label2id"]
id2label = {int(k): v for k, v in maps["id2label"].items()}  # JSON keys are strings

print(tokenized_ds)
print("Classes:", list(label2id.keys()))

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5
    })
})
Classes: ['apt-campaign', 'phishing', 'ransomware', 'supply-chain', 'vulnerability-report']


## Step 2 — Model, tokenizer, collator

In [2]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Step 3 — Metrics

For classification we report **accuracy** (overall) and **macro F1** (averages across classes, robust to imbalance).

In [3]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

## Step 4 — Train

In [4]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="./cls-cti-bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipykernel_57631/2015823129.py:18: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.598291,0.400000,0.280000
2,No log,1.520334,0.400000,0.266667
3,1.529400,1.500664,0.400000,0.266667
4,1.529400,1.446744,0.600000,0.466667
5,1.262900,1.335899,0.600000,0.466667
6,1.262900,1.276853,0.600000,0.466667
7,1.262900,1.245348,0.800000,0.733333
8,1.072800,1.216040,0.600000,0.500000
9,1.072800,1.202709,0.800000,0.733333
10,0.977300,1.207699,0.800000,0.733333


TrainOutput(global_step=20, training_loss=1.2105972528457642, metrics={'train_runtime': 119.0099, 'train_samples_per_second': 1.26, 'train_steps_per_second': 0.168, 'total_flos': 2436926233524.0, 'train_loss': 1.2105972528457642, 'epoch': 10.0})

## Step 5 — Per-class breakdown on validation set

In [5]:
from sklearn.metrics import classification_report

preds_output = trainer.predict(tokenized_ds["validation"])
preds = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print(classification_report(
    y_true, preds,
    target_names=[id2label[i] for i in range(len(id2label))],
    digits=3,
))

                      precision    recall  f1-score   support

        apt-campaign      0.500     1.000     0.667         1
            phishing      1.000     1.000     1.000         1
          ransomware      1.000     1.000     1.000         1
        supply-chain      0.000     0.000     0.000         1
vulnerability-report      1.000     1.000     1.000         1

            accuracy                          0.800         5
           macro avg      0.700     0.800     0.733         5
        weighted avg      0.700     0.800     0.733         5



/home/saqib/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/saqib/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/saqib/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Step 6 — Save

In [6]:
trainer.save_model("./cls-cti-bert-final")
tokenizer.save_pretrained("./cls-cti-bert-final")

('./cls-cti-bert-final/tokenizer_config.json',
 './cls-cti-bert-final/special_tokens_map.json',
 './cls-cti-bert-final/vocab.txt',
 './cls-cti-bert-final/added_tokens.json',
 './cls-cti-bert-final/tokenizer.json')

## Step 7 — Inference with confidence

In [7]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="./cls-cti-bert-final",
    tokenizer="./cls-cti-bert-final",
    top_k=None,  # return all class probabilities
)

tests = [
    "A new variant of LockBit encrypted a hospital's patient record systems.",
    "CVE-2024-12345 in OpenSSL allows remote code execution. Patch now.",
    "APT41 conducted year-long espionage against semiconductor manufacturers.",
    "Employees received fake Microsoft Teams invites leading to credential theft.",
    "A compromised Node.js package in the npm registry harvested SSH keys.",
]

for t in tests:
    scores = clf(t)[0]
    scores.sort(key=lambda x: -x["score"])
    print(t)
    for s in scores[:3]:
        print(f"  {s['label']:<22} {s['score']:.3f}")
    print()

Device set to use cuda:0


A new variant of LockBit encrypted a hospital's patient record systems.
  apt-campaign           0.291
  ransomware             0.248
  supply-chain           0.211

CVE-2024-12345 in OpenSSL allows remote code execution. Patch now.
  vulnerability-report   0.490
  ransomware             0.162
  supply-chain           0.149

APT41 conducted year-long espionage against semiconductor manufacturers.
  apt-campaign           0.301
  ransomware             0.225
  phishing               0.208

Employees received fake Microsoft Teams invites leading to credential theft.
  phishing               0.321
  ransomware             0.210
  apt-campaign           0.189

A compromised Node.js package in the npm registry harvested SSH keys.
  supply-chain           0.268
  apt-campaign           0.225
  ransomware             0.217



## Your turn — exercises

1. **Class weights.** Use the imbalanced dataset from notebook 7 exercise. Pass `class_weights` to a custom Trainer subclass that overrides `compute_loss`. Does minority-class F1 improve?
2. **Freeze the backbone.** Freeze all BERT layers (`param.requires_grad = False`) except the head. How does training speed and final F1 change?
3. **Try a different checkpoint.** Swap `bert-base-uncased` for `distilbert-base-uncased`. Compare size, speed, F1.

## Next notebook

**`09_handling_long_reports.ipynb`** — our examples were all short. Real threat reports are thousands of words. We'll chunk long reports into overlapping windows, classify each chunk, and aggregate.